# Dealing with Conflicts

This file shows how `Larkita` deals with *shift-reduce* and *reduce-reduce* conflicts.  It is a variant of the
notebook `01-Conflicts.ipynb`, which uses `Lark`.
The following grammar is *ambiguous* because it does not specify the precedence of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "*" expr
         | NUMBER
```

## Specification of the Scanner and the Parser

The operators `"+"` and `"*"` occur as *string literals* inside the grammar rules.  `Larkita` uses the string
literals of an alternative as the *head* of the abstract syntax tree that is built for this alternative:
the alternative `expr "+" expr` yields the tuple `('+', lhs, rhs)` and a number yields a pair like `('NUMBER', '42')`.
How the abstract syntax tree is built has no influence on the parse table.

In [ ]:
grammar = r"""
    expr : expr "+" expr
         | expr "*" expr
         | NUMBER

    NUMBER : /0|[1-9][0-9]*/

    %import common.WS
    %ignore WS
"""

## How `Larkita` Reports Its Conflicts

`Larkita` always generates a *canonical LR(1) parser*.  It reports conflicts in two ways:

1. Shift-reduce conflicts are resolved in favour of shifting.  For every grammar with shift-reduce conflicts,
   `Larkita` issues a warning of class `ConflictWarning`, which `Jupyter` shows below the cell that creates the
   parser.  Reduce-reduce conflicts are errors:  they raise an exception of class `GrammarError`.
2. `Larkita` always writes a description of all states of the LR(1) automaton into an *output file*.  Among other
   things, this file lists all conflicts.  The keyword argument `out_file` specifies the name of this file.

In [ ]:
from larkita import Larkita, ConflictWarning, GrammarError

In contrast to `Lark`, whose *default* parser is an *Earley* parser that has no problem at all with an ambiguous
grammar, `Larkita` has no other parsing algorithm.  Hence the conflicts cannot go unnoticed.

In [ ]:
parser = Larkita(grammar, out_file='conflicts.out')

The section `Conflicts` of the output file lists the conflicts.  Every entry names

1. the state and the terminal for which the entry of the action table is ambiguous,
2. the action that has been chosen, and
3. the alternatives:  the state the parser could shift to and the grammar rule that could have been used to reduce
   the symbol stack in this situation.

Note the remark `resolved as shift`:  `Larkita` resolves **every** shift-reduce conflict in favour of shifting.

In [ ]:
with open('conflicts.out') as handle:
    report = handle.read()
print(report[report.index('Conflicts\n'):report.index('States\n')])

The only thing we can do is to declare that we do not want to tolerate any conflicts at all.  The module `warnings`
of `Python` can turn the `ConflictWarning` into an exception.  It is a good idea to develop a grammar this way, for
then a conflict cannot be overlooked.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter('error', ConflictWarning)
    try:
        Larkita(grammar, out_file='conflicts.out')
    except ConflictWarning as e:
        print(e)

## Inspecting the LR(1) States

The section `States` of the output file describes all states of the canonical LR(1) automaton.  Every state lists
its *marked rules* together with their *lookahead* tokens, i.e. the tokens that may follow the rule.  The marked rules
that are added by the closure are marked with a `+`.  Below the marked rules, the actions and the gotos of the state
are listed.  The rules are numbered as in the section `Rules` at the beginning of the file.

Let us look at the states.  `Larkita` marks the position of the parser inside a rule with the bullet `•` that we use
in the lecture notes, and the end of the input is called `$END`.

In the states 5 and 6, which contain a marked rule of the form `expr → expr "+" expr •` or `expr → expr "*" expr •`,
there are two shift-reduce conflicts each.  All of them are resolved in favour of shifting: the only reduce action
that survives in these states is the one for `$END`.  The discarded reduce actions are listed below the actions of
these states.

In [ ]:
print(report[report.index('States\n'):])

## Building an Abstract Syntax Tree

The parser generated by `Larkita` returns the abstract syntax tree as a nested tuple.  Hence we can use the function
`tuple2dot` directly in order to display it.

In [ ]:
%run ../AST2Dot.ipynb

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

The function `test(s)` takes a string `s` as its argument and tries to parse this string.  If all goes well,
the abstract syntax tree is returned and displayed.  If the string can't be parsed, `Larkita` raises an
exception of class `UnexpectedInput`, which we catch in order to print a readable error message.

In [ ]:
from larkita import UnexpectedInput

def test(s):
    try:
        t = parser.parse(s)
    except UnexpectedInput as e:
        print(f'Syntax error:\n{e}')
        return None
    d = tuple2dot(t)
    display(d)
    return t

The next example shows that this parser does not produce the abstract syntax that reflects the precedences
of the arithmetical operators:  the string `2*3+4` is parsed as `2*(3+4)`.

In [ ]:
test('2*3+4')

Due to the fact that all shift-reduce conflicts are resolved in favour of shift, all operators have the
same precedence and effectively associate to the right.

In [ ]:
test('2+3+4')

In [ ]:
test('1+2*3')

The notebook [02-Conflicts-Resolved-larkita.ipynb](02-Conflicts-Resolved-larkita.ipynb) shows how these conflicts can
be removed.  Since `Larkita` does not support operator precedence declarations, the precedences and
associativities of the operators have to be encoded into the grammar by introducing one syntactical
variable per precedence level.